# 02 · Swahili ASR benchmark on GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/claverfred/real_time_swahili_english_interpreter/blob/main/notebooks/02_asr_benchmark.ipynb)

Compares Swahili speech-recognition candidates on **real read speech** (FLEURS `sw_ke` test, 487
recordings, human transcripts) through the deployed pipeline (VAD → ASR → correction list).
Each candidate runs in its own process with `fallback.policy=fail`, so a model that cannot load
is reported as failed instead of being silently replaced. GPU memory is sampled with `nvidia-smi`.

Reported per candidate: WER with 95% bootstrap CI, WER with number words as digits, CER, empty
outputs, decode time p50/p95/p99, real-time factor, peak GPU memory.
The CPU reference (development PC, 10 cores, all 487 recordings): Mollel/ASR-Swahili-Small, beam 5, WER 0.272.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100/L40S are faster). Optional: add a Hugging Face token as the Colab Secret `HF_TOKEN` (key icon on the left); every default model is public, so it is not required.

Results, converted models and data are kept on Google Drive in `MyDrive/swahili_interpreter/`, so later sessions start faster. Nothing is uploaded anywhere else.

In [ ]:
# @title 1. Setup: clone/update the repository, mount Drive, install, fetch data (safe to re-run)
REPO_URL = "https://github.com/claverfred/real_time_swahili_english_interpreter.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
PROFILE = "colab"  # @param {type:"string"}

import os, subprocess, sys
WORKDIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPO = os.path.join(WORKDIR, REPO_URL.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git"))
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "-1", "--format=commit %h  %cd  %s"], capture_output=True, text=True).stdout)

from src.platform import colab
info = colab.bootstrap(REPO, use_drive=USE_DRIVE)
print("persistent root:", info["root"], "(Google Drive)" if info["on_drive"] else "")
for n in info["notes"]:
    print("NOTE:", n)
print("Hugging Face token:", "from Colab Secrets" if info["hf_login"] else "none (fine: all default models are public)")

r = subprocess.run([sys.executable, "scripts/setup_colab.py", "--profile", PROFILE, "--no-smoke"])
stale = colab.restart_required()
if stale:
    print("Restart the runtime (Runtime > Restart session), then run this cell again:", stale)
assert r.returncode == 0, "setup reported a failure (see the table above)"

In [ ]:
# @title 2. Candidates
N = 487  # @param {type:"integer"}
CANDIDATES = [  # (label, model, revision, beam)
    ("Mollel small, beam 5 (cpu/colab default)", "Mollel/ASR-Swahili-Small", "b2bab6c17b39a0d9b17549444535d4018782a958", 5),
    ("Mollel small, beam 1", "Mollel/ASR-Swahili-Small", "b2bab6c17b39a0d9b17549444535d4018782a958", 1),
    ("cdli small afrivoice, beam 5", "cdli/whisper-small-swahili-afrivoice_cv", "main", 5),
    ("OpenAI large-v3-turbo, beam 5", "large-v3-turbo", "main", 5),
    ("OpenAI large-v3, beam 5", "large-v3", "main", 5),
]

In [ ]:
# @title 3. Run (one process per candidate)
runs = []
for label, model, rev, beam in CANDIDATES:
    print(f"\n===== {label} =====")
    r = colab.run_cli(["evaluate", "speech", "--n", str(N), "--no-oracle", "--bootstrap", "500", "--profile", PROFILE,
                       "--set", f"asr.model={model}", "--set", f"asr.revision={rev}", "--set", f"asr.beam_size={beam}",
                       "--set", "fallback.policy=fail"], prefix="eval-speech")
    runs.append({"label": label, **r})
    print(f"-> exit {r['returncode']}, {r['seconds']} s, peak GPU memory {r['gpu_peak_mb']} MB")

In [ ]:
# @title 4. Comparison table (saved as CSV next to the runs)
import pandas as pd
from IPython.display import display
from src.eval.summary import summarize_run

def table(runs, extra_keys=("label", "gpu_peak_mb", "seconds")):
    rows = []
    for r in runs:
        if not r.get("run_dir"):
            rows.append({"label": r.get("label"), "status": f"FAILED (exit {r['returncode']})"})
            continue
        for row in summarize_run(r["run_dir"]):
            rows.append({**{k: r.get(k) for k in extra_keys}, **row, "status": "ok"})
    return pd.DataFrame(rows)

GPU = (colab.environment_report()["cuda"]["gpus"] or [{"name": "cpu"}])[0]["name"].replace(" ", "_")

df = table(runs)
cols = ["label", "status", "n", "wer", "wer_ci95", "wer_numbers_as_digits", "cer", "empty_rate",
        "decode_p50_s", "decode_p95_s", "decode_p99_s", "rtf", "gpu_peak_mb", "device", "asr_model"]
display(df[[c for c in cols if c in df]])
csv = colab.results_dir() / f"colab_asr_comparison_{GPU}.csv"
df.to_csv(csv, index=False)
print("saved", csv)

In [ ]:
# @title 5. Chart: WER with 95% CI
import matplotlib.pyplot as plt
ok = df[df.status == "ok"].copy()
lo_hi = ok.wer_ci95.str.split("-", expand=True).astype(float)
fig, ax = plt.subplots(figsize=(8, 0.5 + 0.45 * len(ok)))
ax.barh(ok.label, ok.wer, xerr=[ok.wer - lo_hi[0], lo_hi[1] - ok.wer], color="#2a78d6", height=0.5, capsize=3)
ax.invert_yaxis(); ax.set_xlabel("WER (lower is better)"); ax.set_title(f"FLEURS sw_ke test, n={N}, {GPU}")
ax.grid(axis="x", color="#e1e0d9"); ax.set_axisbelow(True)
png = colab.results_dir() / f"colab_asr_comparison_{GPU}.png"
fig.savefig(png, dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
RUN_DIRS, EXTRA_FILES, EXPORT_NAME = [r["run_dir"] for r in runs], [csv, png], "asr-benchmark"
# @title Export: ZIP this notebook's runs (JSON, CSV, audio, logs, images, manifests) and download
import time
zip_path = colab.results_dir() / f"{EXPORT_NAME}-{time.strftime('%Y%m%d-%H%M%S')}.zip"
env_files = sorted((colab.results_dir().parent / "env").glob("*"))[-4:]
z = colab.zip_runs([p for p in RUN_DIRS if p], zip_path, extra_files=EXTRA_FILES + env_files)
print(z)
colab.download(zip_path)